# Week 2 : Day 2 Structured Outputs with Pydantic Fundamentals

### Written by: Hafiza Mehak Arif

### Date: 16-June-2026


### Task 1: Pydantic Model Suite


In [ ]:
import pydantic
print(pydantic.__version__)



2.12.5


In [3]:
from pydantic import BaseModel , field_validator ,ValidationError
from typing import List
print ("Import Successful")

Import Successful


#### Person Model


In [9]:
class Person(BaseModel):
    name: str
    age: int
    email: str
    hobbies: List[str]

    @field_validator("age")
    @classmethod
    def validate_age(cls, value):
        if value <= 0:
            raise ValueError("Age must be greater than 0")
        return value

    @field_validator("email")
    @classmethod
    def validate_email(cls, value):
        if "@" not in value:
            raise ValueError("Invalid email format")
        return value

#### Product Model


In [12]:
#Create Product model: id, name, price (float), in_stock (bool), tags (List[str])
class Product(BaseModel):
    name: str
    id: int
    price: float
    in_stock: bool
    tags: list[str]

    @field_validator("price")
    @classmethod
    def price_validator(cls , value):
        if value < 0 :
            raise ValueError("price cannot be negative.")
        return value
    

### Testing with Valid and Invalid data


In [15]:
try:
    person1 = Person(
        name="Mehak",
        age=22,
        email="mehak@example.com",
        hobbies=["Reading", "Coding", "Painting"]
    )

    print(person1)

except ValidationError as e:
    print(e)
    

name='Mehak' age=22 email='mehak@example.com' hobbies=['Reading', 'Coding', 'Painting']


In [11]:
try:
    person2 = Person(
        name="Ali",
        age=-5,
        email="ali@example.com",
        hobbies=["Cricket"]
    )
except ValidationError as e:
    print(e)

1 validation error for Person
age
  Value error, Age must be greater than 0 [type=value_error, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error


In [16]:
try:    
    product_1 = Product(
        id = 1345,
        name=" Phone",
        price= 20.67,
        in_stock= True,
        tags= ["Electronoics"]
    )

    print(product_1)
except ValidationError as e:
    print(e)


name=' Phone' id=1345 price=20.67 in_stock=True tags=['Electronoics']


In [14]:
try:
    product2 = Product(
        id=102,
        name="Phone",
        price=-500,
        in_stock=True,
        tags=["Electronics"]
    )
except ValidationError as e:
    print(e)

1 validation error for Product
price
  Value error, price cannot be negative. [type=value_error, input_value=-500, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error


# Task 2: LLM → Structured Data Pipeline


In [17]:
from pydantic import BaseModel, ValidationError
from typing import List
import json

from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

In [18]:
class Article(BaseModel):
    title: str
    author: str
    published_date: str
    summary: str
    tags: List[str]

In [19]:
load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

In [20]:
structured_llm = llm.with_structured_output(Article)

In [21]:
article_text = """
OpenAI announced a new initiative to improve AI education worldwide.
According to CEO Sam Altman, the program aims to provide free learning
resources to students and educators. The initiative was launched on
June 10, 2026, and focuses on making AI literacy more accessible.
"""

In [22]:
article = structured_llm.invoke(
    f"""
    Extract the following information from the article:
    - Title
    - Author
    - Published Date
    - Summary
    - Tags

    Article:
    {article_text}
    """
)

print(article)

title='OpenAI Launches Initiative to Improve AI Education' author='Unknown' published_date='June 10, 2026' summary='OpenAI announced a new initiative to improve AI education worldwide by providing free learning resources to students and educators.' tags=['AI Education', 'OpenAI', 'AI Literacy']


In [23]:
def extract_article(text):
    try:
        return structured_llm.invoke(
            f"""
            Extract the following information from the article:
            - Title
            - Author
            - Published Date
            - Summary
            - Tags

            Article:
            {text}
            """
        )

    except Exception as e:
        print("Error:", e)
        print("Retrying once...")

        try:
            return structured_llm.invoke(
                f"""
                Extract the following information from the article:
                - Title
                - Author
                - Published Date
                - Summary
                - Tags

                Article:
                {text}
                """
            )

        except Exception as e:
            print("Failed again:", e)

            return Article(
                title="Unknown",
                author="Unknown",
                published_date="Unknown",
                summary="Extraction failed",
                tags=[]
            )

In [24]:
articles = [
    article_text,
    """
    Researchers developed a new solar panel technology that improves efficiency.
    The study was published by scientists at MIT on May 15, 2026.
    """,
    """
    NASA announced plans for its next lunar mission.
    The mission focuses on sustainable human presence on the Moon.
    """,
    """
    Experts warn about rising cybersecurity threats affecting businesses globally.
    Organizations are encouraged to strengthen their defenses.
    """
]

In [25]:
structured_articles = []

for text in articles:
    article = extract_article(text)
    structured_articles.append(article)

In [26]:
json_dataset = [
    article.model_dump()
    for article in structured_articles
]

with open("articles.json", "w", encoding="utf-8") as file:
    json.dump(json_dataset, file, indent=4)

print("Saved", len(json_dataset), "articles to articles.json")

Saved 4 articles to articles.json
